In [1]:
import pandas as pd
import csv
import pandas_gbq
import time
from datetime import datetime
import os
from google.oauth2 import service_account
from google.cloud import bigquery

#### Define file path and set_month

In [4]:
seriatim = pd.DataFrame()
claims = pd.DataFrame()
premium = pd.DataFrame()
annuity = pd.DataFrame()
set_month = '202604'
folder_dest = "I:New Structure/Actuarial New/Database/ACLICO Settlements/"+set_month+"/04-2026 ACL Life Settlement_workbook_FINAL.xlsx"

In [23]:
seriatim = pd.read_excel(folder_dest, sheet_name="CONRE Reserve", dtype=str)

In [24]:
claims = pd.read_excel(folder_dest, sheet_name="APR CONRE CLAIMS", dtype=str)

In [25]:
premiums = pd.read_excel(folder_dest,sheet_name="APR CONRE PREM", dtype=str)

In [26]:
annuity  = pd.read_excel(folder_dest,sheet_name="PNAnn CONRE", dtype=str)

In [27]:
seriatim.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 106896 entries, 0 to 106895
Data columns (total 75 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   pmdno             106896 non-null  object
 1   pmdtype           0 non-null       object
 2   pmdmode           106896 non-null  object
 3   pmdsts            106896 non-null  object
 4   pmdstssub         106896 non-null  object
 5   pmddebit          0 non-null       object
 6   pmdissuedt        106896 non-null  object
 7   pmdissfolo        0 non-null       object
 8   pmddtdwd          106896 non-null  object
 9   pmdprmtot         106896 non-null  object
 10  pmdpaybase        106896 non-null  object
 11  pmdplan           106896 non-null  object
 12  pmdplanver        106896 non-null  object
 13  pmdoverage        106896 non-null  object
 14  pmdacctnew        0 non-null       object
 15  pmdstsexdt        6125 non-null    object
 16  pbfactive         106896 non-null  obj

In [28]:
seriatim.columns = seriatim.columns.map(str.lower)
seriatim.columns = seriatim.columns.map(lambda x : x.replace(" " , "_"))

In [29]:
seriatim.loc[
    seriatim["pmdno"] == "PD-0056023",
    "pmddtdwd"
] = pd.Timestamp("1960-02-01")

seriatim.loc[
    seriatim["pmdno"] == "PD-0032499",
    "pbfdt"
] = pd.Timestamp("2042-02-10")
seriatim.loc[
    seriatim["pmdno"] == "PD-0032499",
    "wrvpbfdt"
] = pd.Timestamp("2042-02-10")

In [30]:
seriatim.head(5)


,pmdno,pmdtype,pmdmode,pmdsts,pmdstssub,pmddebit,pmdissuedt,pmdissfolo,pmddtdwd,pmdprmtot,...,wrvscaleby,wrvplcycnt,wrvridrcnt,wrvprmgros,wrvrcddesc,wrvclass,wrvgrpind,total_reserve,reinsurance_flag,flag
0,AA-0002837,NaN,Q,A,PRMPY,NaN,2000-05-01 00:00:00,NaN,2026-05-01 00:00:00,358.8,...,0,0,1,0,NaN,ORD,I,0,AA-0002837,Y
1,AA-0003018,NaN,M,A,PRMPY,NaN,2000-06-01 00:00:00,NaN,2026-05-01 00:00:00,245.76,...,0,0,1,0,NaN,ORD,I,0,AA-0003018,Y
2,AA-0003018,NaN,M,A,PRMPY,NaN,2005-08-08 00:00:00,NaN,2026-05-01 00:00:00,245.76,...,0,0,1,0,NaN,ORD,I,0,AA-0003018,Y
3,AA-0005897,NaN,M,A,PRMPY,NaN,2000-11-01 00:00:00,NaN,2026-05-01 00:00:00,278.4,...,0,0,1,0,NaN,ORD,I,0,AA-0005897,Y
4,AA-0005897,NaN,M,A,PRMPY,NaN,2000-11-01 00:00:00,NaN,2026-05-01 00:00:00,278.4,...,0,0,1,0,NaN,ORD,I,0,AA-0005897,Y


In [31]:
seriatim = seriatim.astype({
    "pmdissuedt"  : "datetime64[ns]",
    "pbfamt"      : "float64",
    "wrvage"      : "Int64",       # Nullable integer
    "wrvduratn"   : "Int64",       # Nullable integer
    "wrvrsvamt"   : "float64",
    "wrvtsvamt"   : "float64",
    "wrvplcycnt"  : "Int64",       # Nullable integer
    "pmddtdwd"    : "datetime64[ns]",
    "pmdprmtot"   : "float64",
    "pmdpaybase"  : "Int64",       # Nullable integer
    "pmdoverage"  : "Int64",       # Nullable integer
    "pmdstsexdt"  : "datetime64[ns]",
    "wrvpbfdt"    : "datetime64[ns]",
    "pbfamtrpt"   : "float64",
    "pbfprm"      : "float64",
    "pbfprmannl"  : "float64",
    "wrvcalcdt"   : "datetime64[ns]",
    "wrvpbfage"   : "Int64",       # Nullable integer
    "wrvload"     : "float64",
    "wrvloadntg"  : "float64",
    "wrvprmuneg"  : "float64",
    "wrvprmdefg"  : "float64",
    "wrvprmdueg"  : "float64",
    "wrvprmadvg"  : "float64",
    "wrvprmunen"  : "float64",
    "wrvprmdefn"  : "float64",
    "wrvprmduen"  : "float64",
    "wrvrsvfact"  : "float64",
    "wrvrsvysby"  : "float64",
    "wrvnpmfact"  : "float64",
    "wrvdeffact"  : "float64",
    "wrvtsvfact"  : "float64",
    "wrvscaleby"  : "float64",
    "wrvprmgros"  : "float64",
    "pbfdtcd"     : "string"
})

In [32]:
s = seriatim["wrvpbfdt"].astype("string")

# fix invalid Feb 29 (mm/dd/yyyy)
s = s.str.replace(
    r"^02/29/(\d{4})$",
    r"02/28/\1",
    regex=True
)

# now convert safely
seriatim["wrvpbfdt"] = pd.to_datetime(
    s,
    format="%m/%d/%Y",
    errors="coerce"
)

In [33]:
dt_cols = ["pmdissuedt", "pmddtdwd", "pmdstsexdt", "pbfdtcd", "wrvpbfdt", "wrvcalcdt"]

bad = {}
for c in dt_cols:
    try:
        pd.to_datetime(seriatim[c], errors="raise")
    except Exception as e:
        bad[c] = str(e)

bad

{'pbfdtcd': 'Given date string M not likely a datetime present at position 0'}

In [34]:
float_cols = [
    "pbfamt","wrvrsvamt","wrvtsvamt","pmdprmtot","pbfamtrpt","pbfprm","pbfprmannl",
    "wrvload","wrvloadntg","wrvprmuneg","wrvprmdefg","wrvprmdueg","wrvprmadvg",
    "wrvprmunen","wrvprmdefn","wrvprmduen","wrvrsvfact","wrvrsvysby","wrvnpmfact",
    "wrvdeffact","wrvtsvfact","wrvscaleby","wrvprmgros"
]

bad = []
for c in float_cols:
    if c in seriatim.columns:
        s = seriatim[c].astype(str)
        if (s == "00:00:00").any():
            bad.append(c)

bad


[]

In [35]:
seriatim

,pmdno,pmdtype,pmdmode,pmdsts,pmdstssub,pmddebit,pmdissuedt,pmdissfolo,pmddtdwd,pmdprmtot,...,wrvscaleby,wrvplcycnt,wrvridrcnt,wrvprmgros,wrvrcddesc,wrvclass,wrvgrpind,total_reserve,reinsurance_flag,flag
0,AA-0002837,NaN,Q,A,PRMPY,NaN,2000-05-01,NaN,2026-05-01,358.80,...,0.0,0,1,0.0,NaN,ORD,I,0,AA-0002837,Y
1,AA-0003018,NaN,M,A,PRMPY,NaN,2000-06-01,NaN,2026-05-01,245.76,...,0.0,0,1,0.0,NaN,ORD,I,0,AA-0003018,Y
2,AA-0003018,NaN,M,A,PRMPY,NaN,2005-08-08,NaN,2026-05-01,245.76,...,0.0,0,1,0.0,NaN,ORD,I,0,AA-0003018,Y
3,AA-0005897,NaN,M,A,PRMPY,NaN,2000-11-01,NaN,2026-05-01,278.40,...,0.0,0,1,0.0,NaN,ORD,I,0,AA-0005897,Y
4,AA-0005897,NaN,M,A,PRMPY,NaN,2000-11-01,NaN,2026-05-01,278.40,...,0.0,0,1,0.0,NaN,ORD,I,0,AA-0005897,Y
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106891,PD-0351823,NaN,Q,A,PRMPY,NaN,1990-01-08,NaN,2026-05-15,71.16,...,1.0,0,1,0.0,Waiver of Premium Disablement age 60 waiver a...,ORD,I,0,PD-0351823,Y
106892,PD-0351824,NaN,Q,A,PRMPY,NaN,1990-01-08,NaN,2026-07-15,71.16,...,1.0,0,1,0.0,Waiver of Premium Disablement age 60 waiver a...,ORD,I,0,PD-0351824,Y
106893,PD-0352604,NaN,A,A,PRMPY,NaN,1990-03-05,NaN,2027-05-02,83.04,...,1.0,0,1,0.0,Waiver of Premium Disablement age 60 waiver a...,ORD,I,0,PD-0352604,Y
106894,PD-0345315,NaN,A,A,PRMPY,NaN,1988-06-06,NaN,2027-04-22,19.80,...,1.0,0,1,0.0,Waiver of Premium Disablement age 60 waiver a...,ORD,I,0,PD-0345315,Y


In [36]:
seriatim = seriatim.rename(columns={"total_reserve" : "final_rsv"})

In [37]:
seriatim = seriatim.astype({"final_rsv" : "float64"})

In [38]:
seriatim['set_month'] =set_month


In [39]:
seriatim["pmdplan"] = seriatim["pmdplan"].astype(str).str.zfill(6)
seriatim["pmdplanver"] = seriatim["pmdplanver"].astype(str).str.zfill(4)
seriatim["pbfrsvcd1"] = seriatim["pbfrsvcd1"].astype(str).str.zfill(5)

In [40]:
seriatim = seriatim.drop(['reinsurance_flag', 'flag', 'pmdtype', 'reinsur_', 'annuity_reserve', 'look_up_to_reinsured', 
                          'ann_rsv', 'y_only_from_total_reserve', 'lookup_to_reinsured', 'reinsured_flag', 'foxpro', 'reinsured?', 'unnamed:_73', 'years_inforce', '1st_yr_or_renewal', 'policy_count', 
                          'ceded_reserve','catfstren','ceded_rsv', 'originally_missing_val_code', 'change_in_valcode0_from_2/26_version', 'change_in_valcode1_from_2/26_version'], axis=1, errors="ignore")

In [ ]:
# # strings
# str_cols = ["pmdno", "pmdplan", "pmdplanver", "wrvclass", "wrvgrpind"]
# for c in str_cols:
#     if c in seriatim.columns:
#         seriatim[c] = seriatim[c].astype("string")

# # datetimes
# dt_cols = [
#     "pmdissuedt", "pmddtdwd", "pmdstsexdt", "wrvpbfdt", "wrvcalcdt"
# ]
# for c in dt_cols:
#     if c in seriatim.columns:
#         seriatim[c] = pd.to_datetime(seriatim[c], errors="coerce")

# # numerics (example)
# num_cols = ["pbfamt", "wrvage", "wrvduratn", "wrvrsvamt", "wrvtsvamt"]
# for c in num_cols:
#     if c in seriatim.columns:
#         seriatim[c] = pd.to_numeric(seriatim[c], errors="coerce")


In [41]:
for c in ["wrvprmannn", "wrvdsvamt"]:
    seriatim[c] = (
        seriatim[c]
        .replace(r"^\s*$", pd.NA, regex=True)   # blank strings -> NA
        .pipe(pd.to_numeric, errors="coerce")   # '0' -> 0.0, bad -> NaN
    )

In [42]:
seriatim["wrvridrcnt"] = (
    seriatim["wrvridrcnt"]
      .replace(r"^\s*$", pd.NA, regex=True)   # blanks -> NA
      .pipe(pd.to_numeric, errors="coerce")   # '0' -> 0, bad -> NaN
      .astype("Int64")                        # nullable int
)

In [43]:
ts_cols = []
for c in seriatim.columns:
    s = seriatim[c]
    if s.dtype == "object":
        if s.apply(lambda x: isinstance(x, pd.Timestamp)).any():
            ts_cols.append(c)

ts_cols

['pbfdt']

In [44]:
seriatim["pbfdt"] = seriatim["pbfdt"].astype("string")

In [45]:
seriatim.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 106896 entries, 0 to 106895
Data columns (total 73 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   pmdno       106896 non-null  object        
 1   pmdmode     106896 non-null  object        
 2   pmdsts      106896 non-null  object        
 3   pmdstssub   106896 non-null  object        
 4   pmddebit    0 non-null       object        
 5   pmdissuedt  106896 non-null  datetime64[ns]
 6   pmdissfolo  0 non-null       object        
 7   pmddtdwd    106896 non-null  datetime64[ns]
 8   pmdprmtot   106896 non-null  float64       
 9   pmdpaybase  106896 non-null  Int64         
 10  pmdplan     106896 non-null  object        
 11  pmdplanver  106896 non-null  object        
 12  pmdoverage  106896 non-null  Int64         
 13  pmdacctnew  0 non-null       object        
 14  pmdstsexdt  6125 non-null    datetime64[ns]
 15  pbfactive   106896 non-null  object        
 16  pb

In [46]:
seriatim.to_gbq(
    destination_table="lifetemp.seriatim",
    project_id="converge-database",
    if_exists="append",
    progress_bar=True
)


100%|██████████| 1/1 [00:00<?, ?it/s]


In [47]:
claims.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 402 entries, 0 to 401
Columns: 102 entries, Company to FLAG
dtypes: object(102)
memory usage: 320.5+ KB


In [48]:
claims.columns = claims.columns.map(str.lower)
claims.columns = claims.columns.map(lambda x : x.replace(" " , "_"))
claims.columns = claims.columns.map(lambda x : x.replace("/" , "_or_"))

In [49]:
claims.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 402 entries, 0 to 401
Columns: 102 entries, company to flag
dtypes: object(102)
memory usage: 320.5+ KB


In [50]:
claims_selected = claims.iloc[:, [1, 10,12,16,17,45,46, 56, 61,63,68, 70,78, 86, 95,96,97,98,99]]

In [51]:
claims_selected

,cert,adjudication-code,eff-date,date-rec'd,loss-date,service-from-date,service-to-date,submitted,applied-$,net-$-payable,disposition-date,check-date,check-amount,lob,type,cert.1,issue-date,"ann,-ind,-ord",ind_or_group
0,PN-2207560,WHL,19800101,20260401,20260228,20260228,20260228,12540.42,12540.42,12540.42,20260401,20260401,12540.42,108,Preneed,PN-2207560,2022-06-29 00:00:00,ORD,G
1,PN-1907409,WHL,19800101,20260401,20260328,20260328,20260328,14207.25,14207.25,14207.25,20260401,20260401,14207.25,108,Preneed,PN-1907409,2019-11-06 00:00:00,ORD,I
2,PN-2006485,WHL,19800101,20260401,20260324,20260324,20260324,10050.69,10050.69,10050.69,20260401,20260401,10050.69,108,Preneed,PN-2006485,2020-08-28 00:00:00,ORD,I
3,PN-1704505,WHL,19800101,20260401,20260329,20260329,20260329,19244.19,19244.19,19244.19,20260401,20260401,19244.19,108,Preneed,PN-1704505,2017-08-21 00:00:00,ORD,I
4,PN-1700760,WHL,19800101,20260401,20260316,20260316,20260316,10364.67,10364.67,10364.67,20260401,20260401,10364.67,108,Preneed,PN-1700760,2017-02-13 00:00:00,ORD,I
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
397,PN-2100034,WHL,19500101,20260326,20250420,20250420,20250420,1977.64,1977.64,1977.64,20260331,20260331,1977.64,108,Preneed,PN-2100034,2021-01-05 00:00:00,ORD,I
398,AC-3535702,SUR,19500101,20260330,20260330,20260330,20260330,675.15,675.15,675.15,20260331,20260331,675.15,108,Life,AC-3535702,1976-08-23 00:00:00,IND,I
399,PN-2008870,WHL,19500101,20260331,20260314,20260314,20260314,3441.14,3441.14,3441.14,20260331,20260331,3441.14,108,Preneed,PN-2008870,2020-11-11 00:00:00,ORD,I
400,PN-2101375,WHL,19500101,20260331,20260328,20260328,20260328,7979.94,7979.94,7979.94,20260331,20260331,7979.94,108,Preneed,PN-2101375,2021-02-18 00:00:00,ORD,G


In [52]:
claims_selected.columns = claims_selected.columns.map(str.lower)
claims_selected.columns = claims_selected.columns.map(lambda x : x.replace(" " , "_"))
claims_selected.columns = claims_selected.columns.map(lambda x : x.replace("/" , "_"))
claims_selected.columns = claims_selected.columns.map(lambda x : x.replace("-" , "_"))

In [53]:
claims_selected

,cert,adjudication_code,eff_date,date_rec'd,loss_date,service_from_date,service_to_date,submitted,applied_$,net_$_payable,disposition_date,check_date,check_amount,lob,type,cert.1,issue_date,"ann,_ind,_ord",ind_or_group
0,PN-2207560,WHL,19800101,20260401,20260228,20260228,20260228,12540.42,12540.42,12540.42,20260401,20260401,12540.42,108,Preneed,PN-2207560,2022-06-29 00:00:00,ORD,G
1,PN-1907409,WHL,19800101,20260401,20260328,20260328,20260328,14207.25,14207.25,14207.25,20260401,20260401,14207.25,108,Preneed,PN-1907409,2019-11-06 00:00:00,ORD,I
2,PN-2006485,WHL,19800101,20260401,20260324,20260324,20260324,10050.69,10050.69,10050.69,20260401,20260401,10050.69,108,Preneed,PN-2006485,2020-08-28 00:00:00,ORD,I
3,PN-1704505,WHL,19800101,20260401,20260329,20260329,20260329,19244.19,19244.19,19244.19,20260401,20260401,19244.19,108,Preneed,PN-1704505,2017-08-21 00:00:00,ORD,I
4,PN-1700760,WHL,19800101,20260401,20260316,20260316,20260316,10364.67,10364.67,10364.67,20260401,20260401,10364.67,108,Preneed,PN-1700760,2017-02-13 00:00:00,ORD,I
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
397,PN-2100034,WHL,19500101,20260326,20250420,20250420,20250420,1977.64,1977.64,1977.64,20260331,20260331,1977.64,108,Preneed,PN-2100034,2021-01-05 00:00:00,ORD,I
398,AC-3535702,SUR,19500101,20260330,20260330,20260330,20260330,675.15,675.15,675.15,20260331,20260331,675.15,108,Life,AC-3535702,1976-08-23 00:00:00,IND,I
399,PN-2008870,WHL,19500101,20260331,20260314,20260314,20260314,3441.14,3441.14,3441.14,20260331,20260331,3441.14,108,Preneed,PN-2008870,2020-11-11 00:00:00,ORD,I
400,PN-2101375,WHL,19500101,20260331,20260328,20260328,20260328,7979.94,7979.94,7979.94,20260331,20260331,7979.94,108,Preneed,PN-2101375,2021-02-18 00:00:00,ORD,G


In [54]:
claims_selected = claims_selected.rename(columns = {"cert" : "policyno", "net_$_payable" : "net_payable", "ann,_ind,_ord" : "p_type", 
                                                    "ind_or_group": "g_type", "group_or_individual" : "g_type", "issue-date": "issue_date", "check-amount" : "check_amount", 
                                                    "adjudication-code" : "adjudication_code", "date_rec'd" : "date_recorded", "applied_$" : "applied"
                                                   })

In [55]:
claims_selected = claims_selected.rename(columns = {"issued_date" :"issue_date"})

In [56]:
claims_selected = claims_selected.astype({"net_payable" : "float64", "check_amount" : "float64","issue_date" : "datetime64[ns]", "submitted" : "float64", "applied" : "float64" })

In [57]:
claims_selected = claims_selected.drop(['cert.1', 'cert_start'], errors="ignore", axis=1)

In [58]:
claims_selected['set_month'] = set_month
claims_selected

,policyno,adjudication_code,eff_date,date_recorded,loss_date,service_from_date,service_to_date,submitted,applied,net_payable,disposition_date,check_date,check_amount,lob,type,issue_date,p_type,g_type,set_month
0,PN-2207560,WHL,19800101,20260401,20260228,20260228,20260228,12540.42,12540.42,12540.42,20260401,20260401,12540.42,108,Preneed,2022-06-29,ORD,G,202604
1,PN-1907409,WHL,19800101,20260401,20260328,20260328,20260328,14207.25,14207.25,14207.25,20260401,20260401,14207.25,108,Preneed,2019-11-06,ORD,I,202604
2,PN-2006485,WHL,19800101,20260401,20260324,20260324,20260324,10050.69,10050.69,10050.69,20260401,20260401,10050.69,108,Preneed,2020-08-28,ORD,I,202604
3,PN-1704505,WHL,19800101,20260401,20260329,20260329,20260329,19244.19,19244.19,19244.19,20260401,20260401,19244.19,108,Preneed,2017-08-21,ORD,I,202604
4,PN-1700760,WHL,19800101,20260401,20260316,20260316,20260316,10364.67,10364.67,10364.67,20260401,20260401,10364.67,108,Preneed,2017-02-13,ORD,I,202604
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
397,PN-2100034,WHL,19500101,20260326,20250420,20250420,20250420,1977.64,1977.64,1977.64,20260331,20260331,1977.64,108,Preneed,2021-01-05,ORD,I,202604
398,AC-3535702,SUR,19500101,20260330,20260330,20260330,20260330,675.15,675.15,675.15,20260331,20260331,675.15,108,Life,1976-08-23,IND,I,202604
399,PN-2008870,WHL,19500101,20260331,20260314,20260314,20260314,3441.14,3441.14,3441.14,20260331,20260331,3441.14,108,Preneed,2020-11-11,ORD,I,202604
400,PN-2101375,WHL,19500101,20260331,20260328,20260328,20260328,7979.94,7979.94,7979.94,20260331,20260331,7979.94,108,Preneed,2021-02-18,ORD,G,202604


In [59]:
claims_selected = claims_selected.dropna(subset=["policyno"]).reset_index(drop=True)

In [60]:
claims_selected.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 402 entries, 0 to 401
Data columns (total 19 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   policyno           402 non-null    object        
 1   adjudication_code  402 non-null    object        
 2   eff_date           402 non-null    object        
 3   date_recorded      402 non-null    object        
 4   loss_date          402 non-null    object        
 5   service_from_date  402 non-null    object        
 6   service_to_date    402 non-null    object        
 7   submitted          402 non-null    float64       
 8   applied            402 non-null    float64       
 9   net_payable        402 non-null    float64       
 10  disposition_date   402 non-null    object        
 11  check_date         402 non-null    object        
 12  check_amount       402 non-null    float64       
 13  lob                402 non-null    object        
 14  type      

In [61]:
claims_selected.to_gbq("converge-database.lifetemp.claims",
                 if_exists='append',
                  table_schema=None,
                 project_id="converge-database")

100%|██████████| 1/1 [00:00<?, ?it/s]


In [62]:
premiums.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 35 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   Policy Number               8693 non-null   object
 1   CertificateNumber           8693 non-null   object
 2   Insured Name                8693 non-null   object
 3   Transaction Type            8693 non-null   object
 4   State                       8693 non-null   object
 5   Policy Issuance Date        8693 non-null   object
 6   Policy Effective Date       8693 non-null   object
 7   Policy Expiration Date      6651 non-null   object
 8   Transaction Effective Date  8693 non-null   object
 9   Processing Date             8693 non-null   object
 10  Gross Premium               8693 non-null   object
 11  Commission Amount           8693 non-null   object
 12  Net Premium                 8693 non-null   object
 13  EPO Fees                    8693 non-null   obje

In [63]:
premiums

,Policy Number,CertificateNumber,Insured Name,Transaction Type,State,Policy Issuance Date,Policy Effective Date,Policy Expiration Date,Transaction Effective Date,Processing Date,...,Group Holder,Group Holder Code,Paid To Date,Group/Ind,Class of Policy,Status,Issue State,Reinsurance Flag,TYPE,REINSUR
0,000010-0014,AA-0000103,Samuel K Nelson,Renewal,SC,2000-01-01 00:00:00,2000-01-01 00:00:00,2054-01-01 00:00:00,2026-04-06 00:00:00,2026-04-06 00:00:00,...,Atlantic Coast Life Insurance Co,ACLI,2026-05-01 00:00:00,I,ORD,A,SC,Y,AA,Y
1,000010-0014,AA-0000140,Jeanette M Wright,Renewal,SC,2000-02-01 00:00:00,2000-02-01 00:00:00,2045-02-01 00:00:00,2026-04-27 00:00:00,2026-04-27 00:00:00,...,Atlantic Coast Life Insurance Co,ACLI,2026-06-01 00:00:00,I,ORD,A,SC,Y,AA,Y
2,000017-0003,AA-0000318,Calvert Smalls,Renewal,SC,2000-02-01 00:00:00,2000-02-01 00:00:00,2060-02-01 00:00:00,2026-04-06 00:00:00,2026-04-06 00:00:00,...,Atlantic Coast Life Insurance Co,ACLI,2026-05-01 00:00:00,I,ORD,A,SC,Y,AA,Y
3,000010-0014,AA-0000482,Emmanuel S Wright,Renewal,SC,2000-02-01 00:00:00,2000-02-01 00:00:00,2096-02-01 00:00:00,2026-03-30 00:00:00,2026-03-30 00:00:00,...,Atlantic Coast Life Insurance Co,ACLI,2026-06-01 00:00:00,I,ORD,A,SC,Y,AA,Y
4,000010-0014,AA-0000482,Emmanuel S Wright,Renewal,SC,2000-02-01 00:00:00,2000-02-01 00:00:00,2096-02-01 00:00:00,2026-03-30 00:00:00,2026-03-30 00:00:00,...,Atlantic Coast Life Insurance Co,ACLI,2026-06-01 00:00:00,I,ORD,A,SC,Y,AA,Y
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8688,PNGD05-0003,PN-2211502,Joyce A Kirkland,Renewal,LA,2022-09-28 00:00:00,2022-09-28 00:00:00,NaN,2026-04-16 00:00:00,2026-04-16 00:00:00,...,Atlantic Coast Life Insurance Co,ACLI,2026-04-28 00:00:00,I,ORD,A,LA,Y,PN,Y
8689,PNPC10-0006,PN-2211559,Angie L Plaisance,Renewal,LA,2022-09-28 00:00:00,2022-09-28 00:00:00,NaN,2026-04-15 00:00:00,2026-04-15 00:00:00,...,Atlantic Coast Life Insurance Co,ACLI,2026-04-28 00:00:00,I,ORD,A,LA,Y,PN,Y
8690,PNPC10-0006,PN-2211562,Ricky J Plaisance,Renewal,LA,2022-09-28 00:00:00,2022-09-28 00:00:00,NaN,2026-04-15 00:00:00,2026-04-15 00:00:00,...,Atlantic Coast Life Insurance Co,ACLI,2026-04-28 00:00:00,I,ORD,A,LA,Y,PN,Y
8691,PNPD10-0003,PN-2211649,Jhan G Lambert,Renewal,LA,2022-09-29 00:00:00,2022-09-29 00:00:00,NaN,2026-04-20 00:00:00,2026-04-20 00:00:00,...,Atlantic Coast Life Insurance Co,ACLI,2026-04-29 00:00:00,I,ORD,A,LA,Y,PN,Y


In [64]:
prem_insured = premiums[premiums['Reinsurance Flag'] == 'Y']

In [65]:
prem_selected = prem_insured.iloc[:, [1,4,8,10,12,18,28,29]]

In [66]:
prem_selected

,CertificateNumber,State,Transaction Effective Date,Gross Premium,Net Premium,Report Year,Group/Ind,Class of Policy
0,AA-0000103,SC,2026-04-06 00:00:00,18.5,18.5,2026,I,ORD
1,AA-0000140,SC,2026-04-27 00:00:00,25.9,25.9,2026,I,ORD
2,AA-0000318,SC,2026-04-06 00:00:00,17.7,17.7,2026,I,ORD
3,AA-0000482,SC,2026-03-30 00:00:00,0.8,0.8,2026,I,ORD
4,AA-0000482,SC,2026-03-30 00:00:00,5.1,5.1,2026,I,ORD
...,...,...,...,...,...,...,...,...
8688,PN-2211502,LA,2026-04-16 00:00:00,62.5,62.5,2026,I,ORD
8689,PN-2211559,LA,2026-04-15 00:00:00,127.73,127.73,2026,I,ORD
8690,PN-2211562,LA,2026-04-15 00:00:00,131.38,131.38,2026,I,ORD
8691,PN-2211649,LA,2026-04-20 00:00:00,158.28,158.28,2026,I,ORD


In [67]:
prem_selected.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 8693 entries, 0 to 8692
Data columns (total 8 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   CertificateNumber           8693 non-null   object
 1   State                       8693 non-null   object
 2   Transaction Effective Date  8693 non-null   object
 3   Gross Premium               8693 non-null   object
 4   Net Premium                 8693 non-null   object
 5   Report Year                 8693 non-null   object
 6   Group/Ind                   8693 non-null   object
 7   Class of Policy             8693 non-null   object
dtypes: object(8)
memory usage: 611.2+ KB


In [68]:
prem_selected.columns = prem_selected.columns.map(str.lower)
prem_selected.columns = prem_selected.columns.map(lambda x : x.replace(" " , "_"))

In [69]:
prem_selected = prem_selected.rename(columns = {"certificatenumber" : "policyno", "group/ind" : "catindgrp", "class_of_policy" : "catclass", 
                                               "status" : "x", "project_description" : "report_year", "catclass_-_ann,_ind,_ord" : "catclass", "catindgrp_-_individual_or_group" : "catindgrp"})

In [70]:
prem_selected = prem_selected.astype({"gross_premium" : "float64", "net_premium" : "float64","transaction_effective_date" : "datetime64[ns]" })

In [71]:
prem_selected = prem_selected.rename(columns ={"_net_premium_" : "net_premium"} )

In [72]:
prem_selected.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 8693 entries, 0 to 8692
Data columns (total 8 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   policyno                    8693 non-null   object        
 1   state                       8693 non-null   object        
 2   transaction_effective_date  8693 non-null   datetime64[ns]
 3   gross_premium               8693 non-null   float64       
 4   net_premium                 8693 non-null   float64       
 5   report_year                 8693 non-null   object        
 6   catindgrp                   8693 non-null   object        
 7   catclass                    8693 non-null   object        
dtypes: datetime64[ns](1), float64(2), object(5)
memory usage: 611.2+ KB


In [73]:
prem_selected['set_month'] = set_month

In [74]:
prem_selected.to_gbq("converge-database.lifetemp.premiums",
                 if_exists='append',
                  table_schema=None,
                 project_id="converge-database")

100%|██████████| 1/1 [00:00<?, ?it/s]


In [75]:
annuity.columns = annuity.columns.map(str.lower)
annuity.columns = annuity.columns.map(lambda x : x.replace(" " , "_"))

In [76]:
annuity.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1701 entries, 0 to 1700
Data columns (total 26 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   plan        1701 non-null   object
 1   plandesc    1701 non-null   object
 2   policyno    1701 non-null   object
 3   issuedate   1701 non-null   object
 4   premium1    1701 non-null   object
 5   interest1   1701 non-null   object
 6   total1      1701 non-null   object
 7   premium2    1701 non-null   object
 8   interest2   1701 non-null   object
 9   total2      1701 non-null   object
 10  subtotal    1701 non-null   object
 11  discount    1701 non-null   object
 12  rsvamt      1701 non-null   object
 13  errormsg    0 non-null      object
 14  rptcriteri  0 non-null      object
 15  expire0     1701 non-null   object
 16  expire1     1701 non-null   object
 17  expire2     1701 non-null   object
 18  expire3     1701 non-null   object
 19  expire4     1701 non-null   object
 20  expire5 

In [77]:
annuity.head(5)

,plan,plandesc,policyno,issuedate,premium1,interest1,total1,premium2,interest2,total2,...,expire1,expire2,expire3,expire4,expire5,rsvamtnd,indgrp,polno,date,reinsured
0,PNA001,Plan: PNA001 FLEXIBLE DEFERRED ANNUITY,PN -0400352,20040827,7700,885.02,8585.02,7700,915.12,8615.12,...,0,0,0,0,0,8615.12,I,PN-0400352,2004-08-27 00:00:00,PN-0400352
1,PNA001,Plan: PNA001 FLEXIBLE DEFERRED ANNUITY,PN -0500172,20050310,2810.55,2238.65,5049.2,2810.55,2287.96,5098.51,...,0,0,0,0,0,5098.51,I,PN-0500172,2005-03-10 00:00:00,PN-0500172
2,PNA001,Plan: PNA001 FLEXIBLE DEFERRED ANNUITY,PN -0500205,20050331,1500,1270.18,2770.18,1500,1297.23,2797.23,...,0,0,0,0,0,2797.23,I,PN-0500205,2005-03-31 00:00:00,PN-0500205
3,PNA001,Plan: PNA001 FLEXIBLE DEFERRED ANNUITY,PN -0500349,20050613,5999.7,2452.06,8451.76,5999.7,2534.59,8534.29,...,0,0,0,0,0,8534.29,I,PN-0500349,2005-06-13 00:00:00,PN-0500349
4,PNA001,Plan: PNA001 FLEXIBLE DEFERRED ANNUITY,PN -0800204,20080319,940,487.99,1427.99,940,501.93,1441.93,...,0,0,0,0,0,1441.93,I,PN-0800204,2008-03-19 00:00:00,PN-0800204


In [78]:
annuity.policyno = annuity.policyno.map(lambda x : x.replace(" " , ""))
#annuity = annuity.drop(['policyno'],axis=1)
#annuity.rename(columns={"polno":"policyno"}, inplace=True)

In [79]:
annuity = annuity.drop(['reinsured', 'polno'], axis=1, errors ="ignore")

In [80]:
cols_to_convert = annuity.columns[4:22]  # Get column names from index 4 to 30 (0-based index)

# Loop through each column and convert to float64
for col in cols_to_convert:
    annuity[col] = pd.to_numeric(annuity[col], errors='coerce').astype(float)


In [81]:
annuity.date = annuity.date.astype("datetime64")

In [82]:
annuity['set_month'] = set_month

In [83]:
annuity.to_gbq("converge-database.lifetemp.annuity",
                 if_exists='append',
                  table_schema=None,
                 project_id="converge-database")

100%|██████████| 1/1 [00:00<?, ?it/s]


In [84]:
CREDS = '../../converge-database-0331482f2ee5.json'
client = bigquery.Client.from_service_account_json(json_credentials_path=CREDS)

#### Valcode existance check

In [85]:
valcode_char_fix_qry =f'''
UPDATE `lifetemp.seriatim` 
SET pbfrsvcd1 =LPAD(pbfrsvcd1, 5, '0')
WHERE set_month ='{str(set_month)}';
'''
result = client.query(valcode_char_fix_qry)

In [86]:
valcode_check =f'''select distinct pbfrsvcd1 from `lifetemp.seriatim`
WHERE set_month ='{str(set_month)}' AND pbfrsvcd1 NOT IN (select distinct valcode_new from `lifetemp.valcode`);
'''
result = client.query(valcode_check)
rows = [list(row) for row in result.result()]
valcodes= pd.DataFrame(rows, columns=['valcode'])
valcodes

,valcode


### Check for policy existance

In [ ]:
#maybe compare excel or sql lifetemp.reinsured policy vs new seriatim policies

In [87]:
import sys

In [88]:
sys.path.append('../../LDTI')

# Import the function
from LDTI import main_query_run

# Trigger the AVRF analysis
main_query_run("ACL Life")

Starting LDTI run for : ACL Life
starting ACL Life LDTI
None
ACL Life LDTI Finished


In [89]:


# Add directory containing LDTI.py
sys.path.append('../../actuarial-pipelines/reconciliations/acl/life/')

# Import the function
from reconciliation import run_reconciliation

# Trigger the AVRF analysis
run_reconciliation(set_month)


Query making request:  SELECT SUM(net_payable)*0.95 as surr             from `lifetemp.claims`            WHERE set_month = "202604" AND adjudication_code = "SUR"
{'set_month': '202604', 'fieldname': 'surr', 'withdrawal_amount': 32163.941000000003}
Query making request:  SELECT SUM(net_payable)*0.95 as death_i_ord            from `lifetemp.claims`            WHERE set_month = "202604" and adjudication_code IN ("ANN", "WHL", "END") AND g_type ="I" AND p_type = "ORD" 
{'set_month': '202604', 'fieldname': 'death_i_ord', 'withdrawal_amount': 1075631.3535}
Query making request:  SELECT SUM(net_payable)*0.95 as death_g_ord             from `lifetemp.claims`            WHERE set_month = "202604" and adjudication_code IN ("ANN", "WHL", "END") AND g_type ="G" AND p_type = "ORD" 
{'set_month': '202604', 'fieldname': 'death_g_ord', 'withdrawal_amount': 569941.8979999998}
Query making request:  SELECT SUM(net_payable)*0.95 as death_i_ind             from `lifetemp.claims`            WHERE set_mont